# 01 - Build Span-Completion Instruction JSONL

Convert existing full-sequence RefSeq instruction rows into profile-conditioned protein span-completion rows.


In [12]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_dir_for_import(start: Path) -> Path:
    candidates: list[Path] = []
    env_value = os.environ.get("MDNAC_REPO_DIR")
    if env_value:
        candidates.append(Path(env_value).expanduser())
    resolved_start = start.expanduser().resolve()
    candidates.extend([resolved_start, *resolved_start.parents])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError("Could not locate MDNAC repo. Run inside the repo or set MDNAC_REPO_DIR.")


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import bootstrap_notebook

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(json.dumps(RUNTIME, indent=2, ensure_ascii=False))


{
  "repo_dir": "C:\\Users\\Admin\\Desktop\\MDNAC",
  "platform": "Windows",
  "platform_name": "Windows",
  "is_colab": false,
  "is_notebook": true,
  "python": "3.11.15",
  "cuda_available": true,
  "cuda_device_count": 1
}


In [13]:
SOURCE_PART_DIR = REPO_DIR / "data/cache/instruction_train_parts"
SOURCE_PART_GLOB = "instruction_part_*.jsonl"
SOURCE_JSONL = REPO_DIR / "data/compiled/refseq_bacteria_protein/instruction.jsonl"
OUTPUT_DIR = REPO_DIR / "data/compiled/refseq_bacteria_span_completion"
OUTPUT_JSONL = OUTPUT_DIR / "instruction.jsonl"
STATS_JSON = OUTPUT_DIR / "stats.json"

PARAMS = {
    "examples_per_sequence": 4,
    "min_sequence_length": 64,
    "max_sequence_length": 1024,
    "min_mask_length": 8,
    "max_mask_length": 64,
    "left_flank_size": 64,
    "right_flank_size": 64,
    "seed": 42,
    "mask_policies": ["random_span", "n_terminal_span", "c_terminal_span"],
}


def natural_path_key(path: Path) -> tuple:
    import re

    return tuple(int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name))


SOURCE_PARTS = tuple(sorted(SOURCE_PART_DIR.glob(SOURCE_PART_GLOB), key=natural_path_key))
if SOURCE_PARTS:
    SOURCE_PATHS = SOURCE_PARTS
    OUTPUT_PATHS = tuple(OUTPUT_DIR / path.name for path in SOURCE_PATHS)
else:
    SOURCE_PATHS = (SOURCE_JSONL,) if SOURCE_JSONL.is_file() else ()
    OUTPUT_PATHS = (OUTPUT_JSONL,) if SOURCE_PATHS else ()

assert SOURCE_PATHS, (
    f"Missing source instruction JSONL parts: {SOURCE_PART_DIR / SOURCE_PART_GLOB}. "
    f"Fallback single file also missing: {SOURCE_JSONL}"
)
print(json.dumps({
    "source_part_dir": str(SOURCE_PART_DIR),
    "source_paths": [str(path) for path in SOURCE_PATHS],
    "output_dir": str(OUTPUT_DIR),
    "output_paths": [str(path) for path in OUTPUT_PATHS],
    "stats": str(STATS_JSON),
    "params": PARAMS,
}, indent=2))


{
  "source_part_dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts",
  "source_paths": [
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_1.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_2.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_3.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_4.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_5.jsonl"
  ],
  "output_dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_span_completion",
  "output_paths": [
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_span_completion\\instruction_part_1.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_span_completion\\instruction_part_2.jsonl",
    "C:\

In [14]:
from collections import Counter

from libs.protein_completion import convert_instruction_jsonl_to_span_jsonl, validate_jsonl_file

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
part_stats = []
total_valid_rows = 0
skip_reasons = Counter()
mask_length_distribution = Counter()

for source_path, output_path in zip(SOURCE_PATHS, OUTPUT_PATHS, strict=True):
    part_stats_path = OUTPUT_DIR / f"{output_path.stem}.stats.json"
    print(f"converting {source_path.name} -> {output_path.name}")
    current_stats = convert_instruction_jsonl_to_span_jsonl(
        source_path,
        output_path,
        stats_path=part_stats_path,
        **PARAMS,
    )
    current_validation = validate_jsonl_file(output_path)
    assert current_validation["valid_rows"] == current_stats["generated_span_rows"]

    total_valid_rows += current_validation["valid_rows"]
    skip_reasons.update(current_stats["skip_reasons"])
    mask_length_distribution.update({int(k): v for k, v in current_stats["mask_length_distribution"].items()})
    part_stats.append({
        "source_path": str(source_path),
        "output_path": str(output_path),
        "stats_path": str(part_stats_path),
        "validation": current_validation,
        **current_stats,
    })

stats = {
    "source_paths": [entry["source_path"] for entry in part_stats],
    "output_paths": [entry["output_path"] for entry in part_stats],
    "stats_paths": [entry["stats_path"] for entry in part_stats],
    "part_count": len(part_stats),
    "source_rows": sum(entry["source_rows"] for entry in part_stats),
    "accepted_source_rows": sum(entry["accepted_source_rows"] for entry in part_stats),
    "skipped_rows": sum(entry["skipped_rows"] for entry in part_stats),
    "generated_span_rows": sum(entry["generated_span_rows"] for entry in part_stats),
    "invalid_json_rows": sum(entry["invalid_json_rows"] for entry in part_stats),
    "non_object_rows": sum(entry["non_object_rows"] for entry in part_stats),
    "skip_reasons": dict(sorted(skip_reasons.items())),
    "mask_length_distribution": {str(length): count for length, count in sorted(mask_length_distribution.items())},
    "parameters": PARAMS,
    "example_before": next((entry["example_before"] for entry in part_stats if entry.get("example_before")), None),
    "example_after": next((entry["example_after"] for entry in part_stats if entry.get("example_after")), None),
}
validation = {"valid_rows": total_valid_rows, "paths": [entry["output_path"] for entry in part_stats]}
STATS_JSON.write_text(json.dumps(stats, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

assert validation["valid_rows"] == stats["generated_span_rows"]
assert all(Path(path).is_file() for path in stats["output_paths"])
assert STATS_JSON.is_file()

print("parts:", stats["part_count"])
print("source rows:", stats["source_rows"])
print("accepted source rows:", stats["accepted_source_rows"])
print("skipped rows:", stats["skipped_rows"])
print("generated span rows:", stats["generated_span_rows"])
print("skip reasons:", json.dumps(stats["skip_reasons"], indent=2))
print("mask length distribution:", json.dumps(stats["mask_length_distribution"], indent=2))
print("validated rows:", validation["valid_rows"])
print("source-row immutability: checked during conversion")


converting instruction_part_1.jsonl -> instruction_part_1.jsonl
converting instruction_part_2.jsonl -> instruction_part_2.jsonl
converting instruction_part_3.jsonl -> instruction_part_3.jsonl
converting instruction_part_4.jsonl -> instruction_part_4.jsonl
converting instruction_part_5.jsonl -> instruction_part_5.jsonl
parts: 5
source rows: 9967921
accepted source rows: 8958596
skipped rows: 1009325
generated span rows: 35830728
skip reasons: {
  "no_valid_span": 4,
  "too_long": 795261,
  "too_short": 214060
}
mask length distribution: {
  "8": 631963,
  "9": 632101,
  "10": 628085,
  "11": 630724,
  "12": 632142,
  "13": 629562,
  "14": 629519,
  "15": 627110,
  "16": 626531,
  "17": 628691,
  "18": 627390,
  "19": 629360,
  "20": 628326,
  "21": 627547,
  "22": 629076,
  "23": 628479,
  "24": 629520,
  "25": 630965,
  "26": 627844,
  "27": 628311,
  "28": 628805,
  "29": 627346,
  "30": 628486,
  "31": 630053,
  "32": 629886,
  "33": 630413,
  "34": 626746,
  "35": 628501,
  "36": 62

In [15]:
def clipped_payload(payload: dict, *, max_output_chars: int = 96) -> dict:
    clipped = dict(payload)
    output = clipped.get("output")
    if isinstance(output, str) and len(output) > max_output_chars:
        clipped["output"] = output[:max_output_chars] + f"...({len(output)} aa total)"
    return clipped


print("Before row:")
print(json.dumps(clipped_payload(stats["example_before"] or {}), indent=2, ensure_ascii=False))
print("\nAfter row:")
print(json.dumps(clipped_payload(stats["example_after"] or {}), indent=2, ensure_ascii=False))


Before row:
{
  "instruction": "labels protein sequence; description MULTISPECIES: scaffolding protein D [Bacteria].; organism Bacteria; keywords RefSeq; product scaffolding protein D",
  "input": "",
  "output": "MSQVTEQSVRFQTALASIKLIQASAVLDLTEDDFDFLTSNKVWIATDRSRARRCVEACVYGTLDFVGYPRFPAPVEFIAAVIAYYVHPVNIQTACL...(152 aa total)",
  "accession": "WP_000084700.1",
  "organism": "Bacteria",
  "product": "scaffolding protein D",
  "taxonomy": "Unclassified.",
  "keywords": "RefSeq",
  "derived_labels": [
    "protein sequence"
  ]
}

After row:
{
  "instruction": "task protein span completion; labels protein sequence; description MULTISPECIES: scaffolding protein D [Bacteria].; organism Bacteria; keywords RefSeq; product scaffolding protein D",
  "input": "mask_policy random_span; mask_start 14; mask_length 48; left_flank MSQVTEQSVRFQTA; right_flank LDFVGYPRFPAPVEFIAAVIAYYVHPVNIQTACLIMEGAEFTENIINGVERPVKAAELFAFTLR; partial_sequence MSQVTEQSVRFQTA<MASK_48>LDFVGYPRFPAPVEFIAAVIAYYVHPVNIQTACLIMEG

In [16]:
from libs.protein_completion import validate_span_completion_row

before = stats["example_before"]
after = stats["example_after"]
assert before is not None and after is not None, "No sample row was generated. Check skip_reasons in stats.json."
sample_validation = validate_span_completion_row(after, original_sequence=before["output"])

mask_length = sample_validation["mask_length"]
assert after["output"] == before["output"][sample_validation["mask_start"]:sample_validation["mask_end"]]
assert len(after["output"]) == mask_length
assert f"<MASK_{mask_length}>" in after["input"]
assert "<MASK_" in after["input"] and after["input"].count("<MASK_") == 1

print(json.dumps(sample_validation, indent=2))


{
  "mask_start": 14,
  "mask_end": 62,
  "mask_length": 48,
  "left_flank_length": 14,
  "right_flank_length": 64
}
